# Comparaison d'algos en régression

Nous allons comparer les différentes méthodes de régression que nous avons vues jusqu'à présent, les moindres carrés avec toutes les variables, puis les moindres carrés avec les variables sélectionnées par le critère BIC avec un algo backward et par AIC.
Le nombre de bloc de la validation croisée vaut $k=10$ mais peut être modifié par l'utilisateur. Les données s'appellent *don* et la variable d'intérêt $Y$

In [ ]:
!pip install catboost 

In [19]:
import pandas as pd; import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from patsy import dmatrix
#import ols_step_sk
###
from sklearn.linear_model import Ridge, ElasticNet, Lasso
from sklearn.linear_model import RidgeCV, ElasticNetCV, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold
###
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.model_selection import GridSearchCV, KFold, cross_val_score
from sklearn.metrics import mean_squared_error

In [33]:
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split

In [34]:
don = pd.read_csv("https://regression-avec-python.github.io/donnees/ozone.txt", header=0, sep=";",index_col=0)
don.rename(columns={"O3":"Y"},inplace=True)

Codage des variables qualitatives nebu et vent

In [35]:
don = pd.read_csv("https://regression-avec-python.github.io/donnees/ozone_transf.txt", header = 0, sep = ";", index_col=0)
print(don.shape)
don.rename(columns={"maxO3":"Y"},inplace=True)

(1366, 22)


In [36]:
nomsvar = list(don.columns.difference(["Y"]))
print(nomsvar)
#design matrix
formule = "~" + "+".join(nomsvar)
print(formule)
dsX = dmatrix(formule,don)
X = np.asarray(dsX)[:,1:]
Y = don["Y"].to_numpy()

['Ne12', 'Ne15', 'Ne18', 'Ne6', 'Ne9', 'T12', 'T15', 'T18', 'T6', 'T9', 'Vx12', 'Vx15', 'Vx18', 'Vx6', 'Vx9', 'Vy12', 'Vy15', 'Vy18', 'Vy6', 'Vy9', 'maxO3v']
~Ne12+Ne15+Ne18+Ne6+Ne9+T12+T15+T18+T6+T9+Vx12+Vx15+Vx18+Vx6+Vx9+Vy12+Vy15+Vy18+Vy6+Vy9+maxO3v


In [37]:
nb=10 ###nb de blocs
tmp = np.arange(don.shape[0])%nb
rng = np.random.default_rng(seed=1234)
bloc = rng.choice(tmp,size=don.shape[0],replace=False) ### l'indice d'appartenance aux blocs

PREV = pd.DataFrame({"bloc":bloc,"Y":don["Y"],"MCO":0.0,"BIC":0.0,"AIC":0.0,
                    "lasso":0.0,"ridge":0.0,"elastic":0.0,"ridgeGS":0.0,'arbre':0.0,'foret':0.0,
                    "gbm1":0.0,"gbm2":0.0,"gbms1":0.0,"gbms2":0.0,"cat":0.0})

In [38]:
cr = StandardScaler()
kf = KFold(n_splits=10, shuffle=True, random_state=0)
lassocv = LassoCV(cv=kf)
pipe_lassocv = Pipeline(steps=[("cr", cr), ("lassocv", lassocv)])
etape_lassocv = pipe_lassocv.named_steps["lassocv"]
lambdaoptlasso=[]
nopt1=[]
nopt2=[]
enetcv = ElasticNetCV(cv=kf,max_iter=10000)
pipe_enetcv = Pipeline(steps=[("cr", cr), ("enetcv", enetcv)])
ridge = Ridge()
pipe_ridge = Pipeline(steps=[("cr", cr), ("ridge", ridge)])

In [40]:
for i in np.arange(nb):
    print(i)
    Xapp = X[bloc!=i,:]
    Xtest = X[bloc==i,:]
    Yapp = don[bloc!=i]["Y"]
    Ytest = don[bloc==i]["Y"]
    #### reg
    reg = LinearRegression()
    reg.fit(Xapp,Yapp)
    PREV.loc[PREV.bloc==i,"MCO"] = reg.predict(Xtest)
    ### bic
    #inst_reg_bic = ols_step_sk.LinearRegressionSelectionFeatureIC(verbose=1,crit="bic")
    #reg_bic = inst_reg_bic.fit(X=Xapp, y=Yapp)
    #PREV.loc[PREV.bloc==i,"BIC"] = reg_bic.predict(Xtest)
    ### aic
    #inst_reg_aic = ols_step_sk.LinearRegressionSelectionFeatureIC(verbose=1,crit="aic")
    #reg_aic = inst_reg_aic.fit(X=Xapp, y=Yapp)
    #PREV.loc[PREV.bloc==i,"AIC"] = reg_aic.predict(Xtest)
    ###lasso
    pipe_lassocv.fit(Xapp,Yapp)
    PREV.loc[PREV.bloc==i,"lasso"] = pipe_lassocv.predict(Xtest)
    ###elastic net pondération 1/2
    pipe_enetcv.fit(Xapp,Yapp)
    PREV.loc[PREV.bloc==i,"elastic"] = pipe_enetcv.predict(Xtest)
    ###ridge avec le chemin de reg qui vaut 100 le chemin de lasso
    alphasridge = 100*etape_lassocv.alphas_
    ridgecv = RidgeCV(cv=kf,alphas=alphasridge)
    pipe_ridgecv = Pipeline(steps=[("cr", cr), ("ridgecv", ridgecv)])
    pipe_ridgecv.fit(Xapp,Yapp)
    PREV.loc[PREV.bloc==i,"ridge"] = pipe_ridgecv.predict(Xtest)
    ## ridge grid searchparams lambda
    path_ridge = alphasridge   
    param_grid_ridge = {"ridge__alpha": path_ridge}
    cv_ridge = GridSearchCV(pipe_ridge, param_grid_ridge, cv=kf, scoring = "neg_mean_squared_error", n_jobs=3).fit(Xapp, Yapp)
    PREV.loc[PREV.bloc==i,"ridgeGS"] = cv_ridge.predict(Xtest)
    ###arbre
    arbre = DecisionTreeRegressor(min_samples_leaf=5).fit(Xapp,Yapp)
    PREV.loc[PREV.bloc==i,"arbre"] = arbre.predict(Xtest)
    ###foret
    foret = RandomForestRegressor().fit(Xapp,Yapp)
    PREV.loc[PREV.bloc==i,"foret"] = foret.predict(Xtest)
    ###gradient boosting
    params = dict(n_estimators=1000, max_depth=1, learning_rate=0.1, random_state=42)
    gbm_early_stopping = GradientBoostingRegressor(**params,validation_fraction=0.1,n_iter_no_change=10)
    gbm_early_stopping.fit(Xapp,Yapp)
    print(gbm_early_stopping.n_estimators_)
    PREV.loc[PREV.bloc==i,"gbm1"] = gbm_early_stopping.predict(Xtest)
    params = dict(n_estimators=1000, max_depth=2, learning_rate=0.1, random_state=42)
    gbm_early_stopping = GradientBoostingRegressor(**params,validation_fraction=0.1,n_iter_no_change=10)
    gbm_early_stopping.fit(Xapp,Yapp)
    print(gbm_early_stopping.n_estimators_)
    PREV.loc[PREV.bloc==i,"gbm2"] = gbm_early_stopping.predict(Xtest)
    ### stochastic gradient boosting
    params = dict(n_estimators=5000, max_depth=1, learning_rate=0.1, random_state=42,subsample=0.2)
    gbm_early_stopping = GradientBoostingRegressor(**params,validation_fraction=0.1,n_iter_no_change=10)
    gbm_early_stopping.fit(Xapp,Yapp)
    print(gbm_early_stopping.n_estimators_)
    PREV.loc[PREV.bloc==i,"gbms1"] = gbm_early_stopping.predict(Xtest)
    params = dict(n_estimators=5000, max_depth=2, learning_rate=0.1, random_state=42)
    gbm_early_stopping = GradientBoostingRegressor(**params,validation_fraction=0.1,n_iter_no_change=10)
    gbm_early_stopping.fit(Xapp,Yapp)
    print(gbm_early_stopping.n_estimators_)
    PREV.loc[PREV.bloc==i,"gbms2"] = gbm_early_stopping.predict(Xtest)
    ###Catboost
    X_train, X_valid, y_train, y_valid = train_test_split(Xapp,Yapp,test_size=0.1,random_state=42)
    model = CatBoostRegressor(iterations=5000,learning_rate=0.1,subsample=0.1,loss_function="RMSE",
    random_seed=42)
    model.fit(X_train,y_train,
    eval_set=(X_valid, y_valid),
    early_stopping_rounds=10)
    print(model.get_best_iteration())
    PREV.loc[PREV.bloc==i,"cat"] = model.predict(Xtest)

0


124
62


109
62
0:	learn: 21.7968339	test: 23.0002164	best: 23.0002164 (0)	total: 1.56ms	remaining: 7.8s


100:	learn: 9.6248940	test: 11.5030635	best: 11.5030635 (100)	total: 84.2ms	remaining: 4.08s
Stopped by overfitting detector  (10 iterations wait)

bestTest = 11.49202519
bestIteration = 106

Shrink model to first 107 iterations.
106
1


154


...

80
62


0:	learn: 22.2557359	test: 22.1018261	best: 22.1018261 (0)	total: 28ms	remaining: 2m 20s
Stopped by overfitting detector  (10 iterations wait)

bestTest = 10.72872191
bestIteration = 59

Shrink model to first 60 iterations.
59
9


137
73


67


73
0:	learn: 22.2641388	test: 19.5019269	best: 19.5019269 (0)	total: 1.58ms	remaining: 7.89s
Stopped by overfitting detector  (10 iterations wait)

bestTest = 11.99234809
bestIteration = 68

Shrink model to first 69 iterations.
68


On a tout préparé, on peut envoyer

In [41]:
prev = PREV.iloc[:,1:]
np.round((prev.sub(PREV.Y, axis=0)**2).mean(),2)

Y             0.00
MCO         187.51
BIC        7745.43
AIC        7745.43
lasso       187.05
ridge       187.45
elastic     187.20
ridgeGS     187.43
arbre       249.21
foret       155.94
gbm1        160.72
gbm2        156.31
gbms1       164.92
gbms2       156.31
cat         154.87
dtype: float64